# Customer Shopping Behaviour: Who Actually Drives Revenue?

**Author:** Subhranshu Panda &nbsp;|&nbsp; **Type:** Business-question EDA + SQL analytics &nbsp;|&nbsp; **Tools:** Python (pandas), SQL

**The business problem:** a retail stakeholder has 3,900 customer transactions and wants to know where revenue actually comes from, whether subscriptions and discounts are working, and who their most loyal customers are — not just a dashboard of totals.

**Dataset:** 3,900 rows x 18 raw fields (demographics, transaction details, engagement signals, logistics). Cleaned and feature-engineered below, then queried with 10 SQL business questions.

**Headline findings:**
- Male customers drove ~68% of total revenue ($157,890 vs $75,191), tracking the male-skewed customer base.
- Subscription status does **not** predict higher individual spend ($59.49 vs $59.87 avg) — it's a retention lever, not a spend lever.
- 80% of the customer base (3,116 / 3,900) are "Loyal" (11+ prior purchases), and most high-frequency repeat buyers are **not** subscribed — loyalty here is behavioural, not tied to the subscription programme.

*Note on tooling: the full engineering version of this project loads the cleaned data into a real PostgreSQL database and visualises it in Power BI — see the [GitHub repo](https://github.com/SubhranshuPan/customer-shopping-behavior-analysis) for that version, including SQL screenshots and the Power BI dashboard file. This Kaggle notebook runs the same 10 SQL questions against an in-memory SQLite database instead, so it executes end-to-end with a single "Run All" and no external setup.*


## 1. Load the Data

In [ ]:
import os
import glob
import sqlite3

import pandas as pd

# Portable data path: works on Kaggle (dataset mounted under /kaggle/input/...)
# and locally (relative path), without editing anything by hand.
if os.path.exists("/kaggle/input"):
    _matches = glob.glob("/kaggle/input/**/customer_shopping_behavior.csv", recursive=True)
    DATA_PATH = _matches[0] if _matches else "../data/customer_shopping_behavior.csv"
else:
    DATA_PATH = "../data/customer_shopping_behavior.csv"

df = pd.read_csv(DATA_PATH)
df.head()


## 2. Data Cleaning & Feature Engineering

In [ ]:
df.info()
print()
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:
# Fill missing Review Rating with the category-level median (37 rows affected)
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

# Standardize column names -> lowercase snake_case
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

# Engineer age_group (quartile split)
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)

# Engineer purchase_frequency_days from the categorical frequency label
frequency_mapping = {
    'Fortnightly': 14, 'Weekly': 7, 'Monthly': 30, 'Quaterly': 90,
    'Bi-Weekly': 14, 'Annually': 365, 'Every 3 Months': 90,
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

# promo_code_used is a 100% duplicate of discount_applied -> drop it
assert (df['discount_applied'] == df['promo_code_used']).all()
df = df.drop('promo_code_used', axis=1)

print("Cleaned shape:", df.shape)
df.columns.tolist()


**Cleaning summary:** filled 37 missing `review_rating` values with the category median, standardized column names, engineered `age_group` and `purchase_frequency_days`, and dropped `promo_code_used` after confirming it was a 100% duplicate of `discount_applied`.

## 3. Business Questions — SQL Analysis

The cleaned dataframe is loaded into an in-memory SQLite database so every query below is real SQL, executed live (not a screenshot). Techniques used: aggregation, subqueries, `CASE` logic, CTEs, and window functions (`ROW_NUMBER() OVER PARTITION BY`).

In [ ]:
conn = sqlite3.connect(":memory:")
df.to_sql("customer", conn, index=False, if_exists="replace")

def run_q(title, sql):
    print(title)
    result = pd.read_sql(sql, conn)
    display(result)
    return result


In [ ]:
run_q("Q1. Total revenue: male vs. female customers", """
SELECT gender, ROUND(SUM(purchase_amount), 2) AS revenue
FROM customer
GROUP BY gender;
""")

**Reading it:** male customers generate roughly two-thirds of total revenue, tracking the male-skewed customer base rather than a per-customer spend gap.

In [ ]:
run_q("Q2. Customers who used a discount but still spent above the average purchase amount", """
SELECT customer_id, purchase_amount
FROM customer
WHERE discount_applied = 'Yes'
  AND purchase_amount >= (SELECT AVG(purchase_amount) FROM customer);
""")

**Reading it:** a meaningful slice of discount-users are still high spenders, i.e. the discount isn't just being used by price-sensitive low spenders.

In [ ]:
run_q("Q3. Top 5 products by average review rating", """
SELECT item_purchased, ROUND(AVG(review_rating), 2) AS avg_rating
FROM customer
GROUP BY item_purchased
ORDER BY avg_rating DESC
LIMIT 5;
""")

**Reading it:** top-rated products skew toward accessories/footwear (gloves, sandals, boots).

In [ ]:
run_q("Q4. Avg. purchase amount: Standard vs. Express shipping", """
SELECT shipping_type, ROUND(AVG(purchase_amount), 2) AS avg_purchase
FROM customer
WHERE shipping_type IN ('Standard', 'Express')
GROUP BY shipping_type;
""")

**Reading it:** Express shipping customers spend only ~$2 more on average than Standard — shipping preference is not a strong spend signal.

In [ ]:
run_q("Q5. Do subscribers spend more? Avg spend & total revenue, subscribers vs. non-subscribers", """
SELECT subscription_status,
       COUNT(customer_id) AS total_customers,
       ROUND(AVG(purchase_amount), 2) AS avg_spend,
       ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM customer
GROUP BY subscription_status
ORDER BY total_revenue DESC;
""")

**Reading it:** average spend is essentially flat between subscribers and non-subscribers — non-subscribers drive more *total* revenue simply because they're 73% of the base. Subscription is a retention lever, not a spend lever.

In [ ]:
run_q("Q6. Top 5 products by percentage of purchases with a discount applied", """
SELECT item_purchased,
       ROUND(100.0 * SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS discount_rate_pct
FROM customer
GROUP BY item_purchased
ORDER BY discount_rate_pct DESC
LIMIT 5;
""")

**Reading it:** discounting is concentrated in specific products (hats, sneakers, coats near ~50% discount rate), not blanket-applied across the catalogue.

In [ ]:
run_q("Q7. Segment customers into New / Returning / Loyal by purchase history", """
WITH customer_type AS (
    SELECT customer_id, previous_purchases,
           CASE
               WHEN previous_purchases = 1 THEN 'New'
               WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
               ELSE 'Loyal'
           END AS customer_segment
    FROM customer
)
SELECT customer_segment, COUNT(*) AS number_of_customers
FROM customer_type
GROUP BY customer_segment;
""")

**Reading it:** ~80% of the customer base falls into 'Loyal' (11+ previous purchases) — this snapshot captures a mature, retained customer base rather than new acquisition.

In [ ]:
run_q("Q8. Top 3 best-selling products within each category", """
WITH item_counts AS (
    SELECT category, item_purchased, COUNT(customer_id) AS total_orders,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(customer_id) DESC) AS item_rank
    FROM customer
    GROUP BY category, item_purchased
)
SELECT item_rank, category, item_purchased, total_orders
FROM item_counts
WHERE item_rank <= 3
ORDER BY category, item_rank;
""")

**Reading it:** shows which specific SKUs to prioritise for inventory/marketing within each category, rather than just category-level totals.

In [ ]:
run_q("Q9. Are repeat buyers (5+ purchases) more likely to be subscribers?", """
SELECT subscription_status, COUNT(customer_id) AS repeat_buyers
FROM customer
WHERE previous_purchases > 5
GROUP BY subscription_status;
""")

**Reading it:** most high-frequency repeat buyers are **not** subscribed — reinforcing that loyalty here is behavioural, not tied to the subscription programme.

In [ ]:
run_q("Q10. Revenue contribution by age group", """
SELECT age_group, ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM customer
GROUP BY age_group
ORDER BY total_revenue DESC;
""")

**Reading it:** revenue is fairly evenly spread across age groups, with Young Adults contributing the most and Seniors the least — no single age group dominates.

## 4. Key Takeaways

- **Revenue is driven by customer base composition, not per-customer spend differences** — gender and shipping-mode spend gaps are small; the real skew is in who makes up the base.
- **The subscription programme is a retention tool, not a spend lever** — treat it as a churn-reduction metric, not an upsell channel.
- **Discounting is targeted, not blanket** — a small set of products (hats, sneakers, coats) carry most of the discount volume, worth auditing for margin impact.
- **This is a mature, loyal customer base (80% "Loyal" tier)** — acquisition marketing and retention marketing need very different playbooks here.

## 5. Full Engineering Version & Contact

This notebook is the Kaggle-portable version of a larger end-to-end project: **Python (cleaning/feature engineering) → PostgreSQL (this same SQL layer) → Power BI (interactive dashboard)**.

- **Full repo (PostgreSQL + Power BI + SQL screenshots + project report):** [github.com/SubhranshuPan/customer-shopping-behavior-analysis](https://github.com/SubhranshuPan/customer-shopping-behavior-analysis)
- **Contact:** [subhranshu599@gmail.com](mailto:subhranshu599@gmail.com) | [LinkedIn](https://www.linkedin.com/in/subhranshu03/)
